In [1]:
## READS MNQ-TICK from HGT-Export SC chartbook

In [2]:
import pandas as pd
import numpy as np
import datetime as dt 
from pathlib import Path
import time

In [3]:
# Configuration: Style preferences
#plt.style.use('ggplot') # Good default for readability
pd.set_option("display.width", 400)      # total characters per line
pd.set_option("display.max_columns", 30) # prevent wrapping by limiting columns
pd.set_option("display.max_rows", 1000)

In [4]:
import os
os.getcwd()

'/home/vm/pt/hgt-rl/mnq-tick/iteration3'

In [5]:
#symbol = 'mnq'
#SEC = 2

#inFile = f'/mnt/d/SierraChart/data/EXPORT/MNQ-TICK-OSCILLATOR-6SEC.csv'
#outFile= f'data/mnq-tick-oscillator-6sec.pqt'
inFile = f'/mnt/d/SierraChart/data/EXPORT/MNQ_HA_FULL_3SEC.csv'
outFile= f'data/mnq-tick-full-3sec.pqt'

print(inFile, outFile)

/mnt/d/SierraChart/data/EXPORT/MNQ_HA_FULL_3SEC.csv data/mnq-tick-full-3sec.pqt
CPU times: user 80 μs, sys: 5 μs, total: 85 μs
Wall time: 79.6 μs


In [6]:
df = pd.read_csv(inFile)

print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 11003506 entries, 0 to 11003505
Data columns (total 43 columns):
 #   Column         Dtype  
---  ------         -----  
 0   Date           str    
 1   Time           str    
 2   Open           float64
 3   High           float64
 4   Low            float64
 5   Last           float64
 6   Volume         int64  
 7   NumTrades      int64  
 8   OHLCAvg        float64
 9   HLCAvg         float64
 10  HLAvg          float64
 11  BidVol         int64  
 12  AskVol         int64  
 13  JMA            float64
 14  VEL            float64
 15  ZeroLevel      float64
 16  VEL.1          float64
 17  ZeroLevel.1    float64
 18  TopBand        float64
 19  MiddleBand     float64
 20  BottomBand     float64
 21  RSX            float64
 22  TopLevel       float64
 23  BottomLevel    float64
 24  ZeroLevel.2    float64
 25  RSX.1          float64
 26  TopLevel.1     float64
 27  BottomLevel.1  float64
 28  ZeroLevel.3    float64
 29  Momentum       float64


In [7]:
# expand stupid SC date like 2026-7-8 to 2026-07-08
parts = df['Date'].astype(str).str.split('-', expand=True)

year  = parts[0]
month = parts[1].str.zfill(2)
day   = parts[2].str.zfill(2)

df['Date_norm'] = year + '-' + month + '-' + day

# join date and time and convert to wall clock datetime64
df['timestamp'] = pd.to_datetime(
    df['Date_norm'] + ' ' + df['Time'].astype(str),
    utc=False,            # keep as wall clock, no timezone
    errors='raise'        # or 'coerce' if you want bad rows as NaT
)

# remove temp columns
df = df.drop(columns=['Date', 'Date_norm', 'Time'])

# make timestamp first column
col = df.pop('timestamp')
df.insert(0, 'timestamp', col)

print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 11003506 entries, 0 to 11003505
Data columns (total 42 columns):
 #   Column         Dtype         
---  ------         -----         
 0   timestamp      datetime64[us]
 1   Open           float64       
 2   High           float64       
 3   Low            float64       
 4   Last           float64       
 5   Volume         int64         
 6   NumTrades      int64         
 7   OHLCAvg        float64       
 8   HLCAvg         float64       
 9   HLAvg          float64       
 10  BidVol         int64         
 11  AskVol         int64         
 12  JMA            float64       
 13  VEL            float64       
 14  ZeroLevel      float64       
 15  VEL.1          float64       
 16  ZeroLevel.1    float64       
 17  TopBand        float64       
 18  MiddleBand     float64       
 19  BottomBand     float64       
 20  RSX            float64       
 21  TopLevel       float64       
 22  BottomLevel    float64       
 23  ZeroLevel.2    f

In [8]:
'''
RangeIndex: 5512771 entries, 0 to 5512770
Data columns (total 23 columns):
 #   Column        Dtype         
---  ------        -----         
 0   timestamp     datetime64[us]
 1   Open          float64       
 2   High          float64       
 3   Low           float64       
 4   Last          float64       
 5   JMA           float64     on HA Last   
 6   VEL           float64     on JMA  
 7   VEL.1         float64     adptive VEL on JMA
 8   TopBand       float64     bollinger on adaptive VEL  
 9   MiddleBand    float64     +  
 10  BottomBand    float64     +  
 11  RSX           float64     on Open (?)  
 12  RSX.1         float64     TICK RSX on Open (?)  
 13  Momentum      float64     on JMA len=2  
 14  Momentum.1    float64     on Momentum len=3  
 15  TopBand.1     float64     bollinger on Momentum.1  
 16  MiddleBand.1  float64     +  
 17  BottomBand.1  float64     +  
 18  Momentum.2    float64     on TICK JMA len=2
 19  Momentum.3    float64     on Momentum.2 len=3  
 20  TopBand.2     float64     bollinger on Momentum.3  
 21  MiddleBand.2  float64     +  
 22  BottomBand.2  float64     +

 #   Column           Dtype         
---  ------           -----         
 0   timestamp        datetime64[us]
 1   Open             float64       
 2   High             float64       
 3   Low              float64       
 4   Last             float64       
 5   JMA              float64       on HA Last   
 6   VEL              float64       on JMA  
 7   adpVEL           float64       adptive VEL on JMA
 8   bolTopAdpVEL     float64       bollinger on adaptive VEL  
 9   bolMidAdpVEL     float64       +  
 10  bolBotAdpVEL     float64       +  
 11  RSX              float64       on Open (?)  
 12  tickRSX          float64       TICK RSX on Open (?)  
 13  jmaD1            float64       momentum on JMA len=2  
 14  jmaD2            float64       momentum on jmaD1 len=3  
 15  bolTopJmaD2      float64       bollinger on jmaD2  
 16  bolMidJmaD2      float64       +  
 17  bolBotJmaD2      float64       +  
 18  tickJmaD1        float64       momentum on TICK JMA len=2
 19  tickJmaD2        float64       on Momentum on tickJmaD1 len=3  
 20  bolTopTickJmaD2  float64       bollinger on tickJmaD2  
 21  bolMidTickJmaD2  float64       +  
 22  bolBotTickJmaD2  float64       +
 23  tickJMA          float64       TICK JMA on HA Last
 '''

df.drop(columns=['Volume','NumTrades','OHLCAvg','HLCAvg','HLAvg','BidVol','AskVol','ZeroLevel','ZeroLevel.1','TopLevel','BottomLevel','ZeroLevel.2','TopLevel.1','BottomLevel.1','ZeroLevel.3','Line','Line.1','Line.2'], inplace=True) 

df.rename(columns={
    'VEL.1': 'adpVEL',
    'TopBand': 'bolTopAdpVEL',
    'MiddleBand': 'bolMidAdpVEL',
    'BottomBand': 'bolBotAdpVEL',    
    'RSX.1': 'tickRSX',
    'Momentum': 'jmaD1',
    'Momentum.1': 'jmaD2',
    'TopBand.1': 'bolTopJmaD2',
    'MiddleBand.1': 'bolMidJmaD2',
    'BottomBand.1': 'bolBotJmaD2',
    'Momentum.2': 'tickJmaD1',
    'Momentum.3': 'tickJmaD2',
    'TopBand.2': 'bolTopTickJmaD2',
    'MiddleBand.2': 'bolMidTickJmaD2',
    'BottomBand.2': 'bolBotTickJmaD2',
    'JMA.1': 'tickJMA',
}, inplace=True)

print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 11003506 entries, 0 to 11003505
Data columns (total 24 columns):
 #   Column           Dtype         
---  ------           -----         
 0   timestamp        datetime64[us]
 1   Open             float64       
 2   High             float64       
 3   Low              float64       
 4   Last             float64       
 5   JMA              float64       
 6   VEL              float64       
 7   adpVEL           float64       
 8   bolTopAdpVEL     float64       
 9   bolMidAdpVEL     float64       
 10  bolBotAdpVEL     float64       
 11  RSX              float64       
 12  tickRSX          float64       
 13  jmaD1            float64       
 14  jmaD2            float64       
 15  bolTopJmaD2      float64       
 16  bolMidJmaD2      float64       
 17  bolBotJmaD2      float64       
 18  tickJmaD1        float64       
 19  tickJmaD2        float64       
 20  bolTopTickJmaD2  float64       
 21  bolMidTickJmaD2  float64       
 22  bol

In [9]:
print(f'monotonic: {df["timestamp"].is_monotonic_increasing}')

df['date'] = df['timestamp'].dt.normalize()

filtered_days = [
   '2022-01-17', '2022-07-04', '2022-11-24', '2023-01-16',
   '2023-02-20', '2023-04-07', '2023-05-29', '2023-06-19',
   '2023-07-04', '2023-09-04', '2023-11-23', '2024-02-19',
   '2024-05-27', '2024-06-19', '2024-07-04', '2024-11-28',
   '2025-01-09', '2025-07-04', '2025-09-01', '2025-11-27',
   '2026-04-03', '2026-07-09',               '2025-11-28'
]

# before 1168
days = df["date"]
all_days = days.drop_duplicates().sort_values()
print("Unique days before:", len(all_days))

df = df[
    ~df["timestamp"].dt.normalize().isin(pd.to_datetime(filtered_days))
]

# after -23 = 1145
days = df["date"]
all_days = days.drop_duplicates().sort_values()
print(" Unique days after:", len(all_days))

#

print(df.info())
print(df.head())

monotonic: True
Unique days before: 1167
 Unique days after: 1145
<class 'pandas.DataFrame'>
Index: 10897778 entries, 0 to 11003505
Data columns (total 25 columns):
 #   Column           Dtype         
---  ------           -----         
 0   timestamp        datetime64[us]
 1   Open             float64       
 2   High             float64       
 3   Low              float64       
 4   Last             float64       
 5   JMA              float64       
 6   VEL              float64       
 7   adpVEL           float64       
 8   bolTopAdpVEL     float64       
 9   bolMidAdpVEL     float64       
 10  bolBotAdpVEL     float64       
 11  RSX              float64       
 12  tickRSX          float64       
 13  jmaD1            float64       
 14  jmaD2            float64       
 15  bolTopJmaD2      float64       
 16  bolMidJmaD2      float64       
 17  bolBotJmaD2      float64       
 18  tickJmaD1        float64       
 19  tickJmaD2        float64       
 20  bolTopTickJmaD2 

In [10]:
df.to_parquet(outFile, index=False)

In [ ]:
print(df["timestamp"].dt.time.min())    
x = 1/0

In [14]:
# find out why one is larger than the other by 1 day
outFile1= f'data/mnq-tick-full-3sec.pqt'
outFile2= f'data/mnq-ohlc-raw-3sec.pqt'

df1 = pd.read_parquet(outFile1)
df2 = pd.read_parquet(outFile2)

days1 = df1['date'].unique()
days2 = df2['date'].unique()

print(len(days1), len(days2))

missing = [x for x in days2 if x not in days1]
print(missing)

1145 1145
[]
